In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

# Download and load the dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

# Fill missing values with zeros
df = df.fillna(0)

# Split the dataset
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# Separate target variable from features
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

# Turn dataframes into matrices using DictVectorizer
dv = DictVectorizer(sparse=True)

train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

# Q1

In [2]:
# Train a decision tree regressor with max_depth=1
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# Inspect the feature used for the split
feature_names = dv.get_feature_names_out()
split_feature_index = dt.tree_.feature[0]
split_threshold = dt.tree_.threshold[0]

print(f"Feature used for split: {feature_names[split_feature_index]}")
print(f"Threshold: {split_threshold}")

Feature used for split: model_year
Threshold: 1997.5


# Q2

In [3]:
# Train a Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Predict on the validation data and calculate RMSE
y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)

print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 1.8332


# Q3

In [4]:
scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)

    scores.append((n, round(rmse, 3)))
    print(f"n_estimators={n:3d} | RMSE={rmse:.3f}")

n_estimators= 10 | RMSE=1.833
n_estimators= 50 | RMSE=1.774
n_estimators=100 | RMSE=1.770
n_estimators=150 | RMSE=1.769


# Q4

In [5]:
depths = [10, 15, 20, 25]
n_estimators_list = [10, 50, 100, 150]

depth_scores = {}

for d in depths:
    scores = []
    for n in n_estimators_list:
        rf = RandomForestRegressor(
            max_depth=d,
            n_estimators=n,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        scores.append(rmse)

    mean_rmse = np.mean(scores)
    depth_scores[d] = mean_rmse
    print(f"max_depth={d:2d} | Mean RMSE: {mean_rmse:.4f}")

best_depth = min(depth_scores, key=depth_scores.get)
print(f"\nBest max_depth: {best_depth}")

max_depth=10 | Mean RMSE: 1.7544
max_depth=15 | Mean RMSE: 1.7842
max_depth=20 | Mean RMSE: 1.7868
max_depth=25 | Mean RMSE: 1.7863

Best max_depth: 10


# Q5

In [6]:
# Train the Random Forest Regressor
rf_q5 = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf_q5.fit(X_train, y_train)

# Extract and match feature importances to feature names
feature_names = dv.get_feature_names_out()
importances = rf_q5.feature_importances_

# Create a sorted dataframe or list of importances
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feature_importance_df.head(10))

                         feature  importance
9                     model_year    0.296107
15                vehicle_weight    0.203313
6             fuel_type=Gasoline    0.160712
8                     horsepower    0.078271
4            engine_displacement    0.061281
0                   acceleration    0.054998
14                    origin=USA    0.042152
2   drivetrain=Front-wheel drive    0.020062
11                     num_doors    0.019401
5               fuel_type=Diesel    0.016571


# Q6

In [8]:
# Create DMatrix for train and validation
features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'val')]

# Test eta = 0.3
xgb_params_1 = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_3 = xgb.train(xgb_params_1, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=10)

# Test eta = 0.1
xgb_params_2 = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_1 = xgb.train(xgb_params_2, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=10)

[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646
[0]	train-rmse:2.74739	val-rmse:2.78635
[10]	train-rmse:1.80592	val-rmse:1.93152
[20]	train-rmse:1.55847	val-rmse:1.75076
[30]	train-rmse:1.47544	val-rmse:1.71667
[40]	train-rmse:1.43074	val-rmse:1.71289
[50]	train-rmse:1.40368	val-rmse:1.71358
[60]	train-rmse:1.37884	val-rmse:1.71688
[70]	train-rmse:1.36283	val-rmse:1.71919
[80]	train-rmse:1.34578	val-rmse:1.72100
[90]	train-rmse:1.32546	val-rmse:1.72303
[99]	train-rmse:1.31192	val-rmse:1.72481
